# A19 lab: siting with map layers, in NumPy

*Energy Systems, Step by Step* · Analyst–Modeler · optional lab for lesson A19 · about 60 minutes, practice included.

**Prairie City is fictional; its numbers are illustrative.** You will rebuild the lesson's map of the region around it from the same seed, and check every number against the page with `assert` cells:

1. The seeded recipe: a 32-bit hash, value noise, and nine layers on a grid of 1 km cells.
2. Homes and setbacks: the share of each cell outside a setback.
3. Exclusion masks and the land left for a solar farm.
4. Criterion scores, the weighted overlay and the best 2 km × 2 km blocks.
5. Weights as value choices: 500 random weightings.
6. A cost surface and Dijkstra's least-cost path for the wind farm's line.
7. The sites passed to a cost model; the wind farm and the data-center campus.
8. Two limits: 2 km cells and a dated homes layer.
9. Cleaning raw layers before a screen, on simulated raw layers: coordinates, missing values, alignment.
10. With real data: the public layers a real screen would use (reading only; nothing is downloaded here).
11. Practice, including a supply curve.

**How to run it.** Put this notebook and `a19-gis-siting.json` (the *Lesson data* button on the lesson page) in one folder. In Google Colab: *File → Upload notebook*, then upload the data file with the folder icon on the left, then *Runtime → Run all*. Locally: Python 3.9 or later with `numpy`; `matplotlib` draws the maps if you have it.

**How long it takes.** A few seconds on a 2025 laptop (the layers are built in plain Python loops, so that every number is computed in the same order as on the page).

In [ ]:
import heapq
import json
import math
from pathlib import Path

import numpy as np

assert Path("a19-gis-siting.json").exists(), "a19-gis-siting.json not found: put it next to this notebook (in Colab, upload it with the folder icon)."
D = json.load(open("a19-gis-siting.json"))
REF = D["reference"]
G = D["grid"]
# the campus's contract MW belongs to the shared Prairie City data (prairie-city.json); the page's test copies it here
D["types"]["dc"]["mw"] = REF["cityDataCenter"]["contractMW"]
W, H, SEED = G["W"], G["H"], G["seed"]
N = W * H
try:
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except ImportError:
    HAVE_PLT = False

def close(a, b, tol=2e-6):
    # the reference block keeps 6 decimals; a sum over 1,320 cells may differ in its last bits
    return abs(a - b) <= tol * max(1.0, abs(b))

print(D["fictional"])
print(W, "x", H, "cells of", G["cellKm"], "km; seed", SEED)

## 1. The seeded recipe

Every layer comes from a seed and a few stated rules, so anyone can rebuild it. The page uses **mulberry32**, a small 32-bit generator; the cell below ports it with the 32-bit arithmetic made explicit (a generator like this is fine for teaching; for your own work use NumPy's `default_rng`).

**Value noise.** A random number sits on each corner of a lattice (from a hash of the corner's coordinates and the seed); between corners the value is blended smoothly with $s = f^2 (3 - 2f)$. Two such patterns at different spatial scales, the second twice as fine and half as strong, give slow swells with some texture. Only $+ - \times \div$, square roots and `floor` are used, so Python and JavaScript compute the very same numbers.

In [ ]:
MASK = 0xFFFFFFFF

def imul(a, b):
    # JavaScript's Math.imul: the low 32 bits of a * b (as an unsigned number here)
    return (a * b) & MASK

def mulberry32(seed):
    a = seed & MASK
    def draw():
        nonlocal a
        a = (a + 0x6D2B79F5) & MASK
        t = imul(a ^ (a >> 15), a | 1)
        t ^= (t + imul(t ^ (t >> 7), t | 61)) & MASK
        return ((t ^ (t >> 14)) & MASK) / 4294967296
    return draw

def lattice(ix, iy, seed):
    s = imul(ix, 0x27d4eb2d) ^ imul(iy, 0x165667b1) ^ imul(seed, 0x2c1b3c6d)
    return mulberry32(s)()

def noise(x, y, scale, seed):
    u, v = x / scale, y / scale
    ix, iy = math.floor(u), math.floor(v)
    fx, fy = u - ix, v - iy
    sx, sy = fx * fx * (3 - 2 * fx), fy * fy * (3 - 2 * fy)
    a, b = lattice(ix, iy, seed), lattice(ix + 1, iy, seed)
    c, d = lattice(ix, iy + 1, seed), lattice(ix + 1, iy + 1, seed)
    top, bot = a + (b - a) * sx, c + (d - c) * sx
    return top + (bot - top) * sy

def fbm(x, y, scale, seed):
    return (noise(x, y, scale, seed) + 0.5 * noise(x, y, scale / 2, seed + 1)) / 1.5

vals = [noise(x * 0.37, y * 0.41, 5, 99) for x in range(100) for y in range(60)]
assert min(vals) >= 0 and max(vals) < 1
print("noise range", round(min(vals), 3), "to", round(max(vals), 3))

### The layers

The recipe in the data file, in words (x runs east, y south, in km from the north-west corner; a cell's centre is at $(i + 0.5, j + 0.5)$):

- **River** along $y_r(x)$, flood zone within 1 to 3 km of it; **city** where a stretched distance from the centre plus a wobble is at most 6 km (core within 2.6, rural fringe to 9.5); an **industrial park** block.
- **Slope** (% within a cell): gentle prairie, bluffs just outside the flood zone, the flanks of a western ridge, a few hills.
- **Wind** at 100 m: faster to the west and on the ridge. **Sunshine**: nearly even. **Land price**: higher next to the city.
- **Protected land**: two ellipses with a seeded edge and a riparian strip. **Homes**: one seeded draw of three numbers per cell, row by row.

In [ ]:
CITY, RIV, SL, WI, SU, LA, IN, HM = D["city"], D["river"], D["slope"], D["wind"], D["sun"], D["land"], D["industrial"], D["homes"]

def river_y(x):
    return RIV["y0"] + RIV["amp"] * (fbm(x, 0, RIV["scale"], SEED + RIV["salt"]) - 0.5) + RIV["tilt"] * (x - RIV["tiltAt"])

def flood_half(x):
    f = RIV["flood"]
    return f["base"] + f["amp"] * fbm(x, 3, f["scale"], SEED + f["salt"])

def ridge_x(y):
    g = SL["ridge"]
    return g["x0"] + g["amp"] * (fbm(0, y, g["scale"], SEED + g["salt"]) - 0.5)

def city_q(x, y):
    dx, dy = (x - CITY["cx"]) / CITY["ax"], (y - CITY["cy"]) / CITY["ay"]
    return math.sqrt(dx * dx + dy * dy) + CITY["wobble"] * (fbm(x, y, 4, SEED + 3) - 0.5)

def in_ellipse(p, x, y):
    dx, dy = (x - p["cx"]) / p["rx"], (y - p["cy"]) / p["ry"]
    return dx * dx + dy * dy + 0.5 * (noise(x, y, 2, SEED + p["salt"]) - 0.5) <= 1

def build(newer=False):
    L = {k: [0] * N for k in ["water", "flood", "built", "ind", "prot", "district", "subc"]}
    for k in ["slope", "wind", "sun", "land", "q"]:
        L[k] = [0.0] * N
    yr = [river_y(i + 0.5) for i in range(W)]
    wf = [flood_half(i + 0.5) for i in range(W)]
    xr = [ridge_x(j + 0.5) for j in range(H)]
    rg, hl = SL["ridge"], SL["hills"]
    for j in range(H):
        for i in range(W):
            n, x, y = j * W + i, i + 0.5, j + 0.5
            dyr = abs(y - yr[i])
            q = city_q(x, y); L["q"][n] = q
            if dyr < 0.5: L["water"][n] = 1
            elif dyr <= wf[i]: L["flood"][n] = 1
            if not L["water"][n] and q <= CITY["built"]: L["built"][n] = 1
            if IN["i0"] <= i <= IN["i1"] and IN["j0"] <= j <= IN["j1"] and not L["water"][n]:
                L["ind"][n] = 1; L["built"][n] = 0
            L["district"][n] = 3 if q <= CITY["core"] else 2 if q <= CITY["built"] else 1 if q <= CITY["fringe"] else 0
            sl = SL["base"] + SL["baseAmp"] * fbm(x, y, SL["baseScale"], SEED + SL["baseSalt"])
            dfz = dyr - wf[i]
            if 0 <= dfz < SL["bluffKm"]: sl += SL["bluff"] * (1 - dfz / SL["bluffKm"])
            dr = abs(x - xr[j])
            if dr < rg["half"]: sl += rg["peak"] * 4 * (dr / rg["half"]) * (1 - dr / rg["half"])
            hv = fbm(x, y, hl["scale"], SEED + hl["salt"])
            if hv > hl["from"] and x < hl["xMax"]: sl += hl["gain"] * (hv - hl["from"])
            L["slope"][n] = sl
            rx = (x - xr[j]) / WI["ridgeKm"]
            L["wind"][n] = WI["east"] + WI["westGain"] * (1 - x / W) + WI["ridgeGain"] / (1 + rx * rx) + WI["amp"] * (fbm(x, y, WI["scale"], SEED + WI["salt"]) - 0.5)
            L["sun"][n] = SU["base"] + SU["southGain"] * (y / H) + SU["amp"] * (fbm(x, y, SU["scale"], SEED + SU["salt"]) - 0.5)
            dc = max(0, q - CITY["built"]) / LA["cityKm"]
            L["land"][n] = LA["base"] + LA["amp"] * fbm(x, y, LA["scale"], SEED + LA["salt"]) + LA["cityGain"] / (1 + dc * dc)
    for k, p in enumerate(D["protected"]):
        for j in range(H):
            for i in range(W):
                n, x, y = j * W + i, i + 0.5, j + 0.5
                if L["built"][n] or L["ind"][n]: continue
                hit = ((L["flood"][n] or L["water"][n]) and p["x0"] <= x <= p["x1"]) if p.get("riparian") else in_ellipse(p, x, y)
                if hit and not L["prot"][n]: L["prot"][n] = k + 1
    homes, rng = [], mulberry32(SEED + HM["salt"])
    for j in range(H):
        for i in range(W):
            n = j * W + i
            r1, r2, r3 = rng(), rng(), rng()
            is_open = not L["water"][n] and not L["built"][n] and not L["ind"][n] and not L["prot"][n]
            p = HM["pFringe"] if L["district"][n] == 1 else HM["pRural"]
            if is_open and r1 < p: homes.append((i + 0.15 + 0.7 * r2, j + 0.15 + 0.7 * r3, False))
    NW, rng2, n_new = HM["newer"], mulberry32(SEED + HM["newer"]["salt"]), 0
    for j in range(H):
        for i in range(W):
            n = j * W + i
            r1, r2, r3 = rng2(), rng2(), rng2()
            grow = (not L["water"][n] and not L["built"][n] and not L["ind"][n] and not L["prot"][n] and not L["flood"][n]
                    and NW["i0"] <= i <= NW["i1"] and NW["j0"] <= j <= NW["j1"])
            if grow and r1 < NW["pGrow"]:
                n_new += 1
                if newer: homes.append((i + 0.15 + 0.7 * r2, j + 0.15 + 0.7 * r3, True))
    for s in D["substations"]:
        L["subc"][math.floor(s["y"]) * W + math.floor(s["x"])] = 1
    L["homes"], L["n_newer"], L["river_y"] = homes, n_new, yr
    return L

R = build()
print(len(R["homes"]), "homes in the 2019 layer;", R["n_newer"], "built since")

### Distances

A setback is smaller than a 1 km cell, so each cell is sampled at 4 × 4 points, 250 m apart.

- **The share left.** The share of points farther than the setback from every home and every built-up cell *estimates* the share of the cell left.
- **What it can miss.** A buffer narrower than the spacing can fall between the points: at the solar farm's 61 m setback, 127 of the 159 cells with a home keep all 16 points (the next cell counts them). To check a site's compliance, use a finer mask or the buffers themselves as shapes (vector geometry).
- **The score's distance.** The cell's centre gives the distance used by the "away from homes" score.
- **The search.** Homes and built-up cells are searched within 3 cells; anything farther counts as 3 km, beyond every setback and score ramp.

In [ ]:
SUB, CAP = G["sub"], 3.0

def dist_square(px, py, i, j):
    dx = max(i - px, 0, px - (i + 1)); dy = max(j - py, 0, py - (j + 1))
    return math.sqrt(dx * dx + dy * dy)

def distances(L):
    bucket = [[] for _ in range(N)]
    for h in L["homes"]:
        bucket[math.floor(h[1]) * W + math.floor(h[0])].append(h)
    built = L["built"]
    def near_home(ci, cj, px, py):
        best = CAP
        for bj in range(max(0, cj - 3), min(H - 1, cj + 3) + 1):
            for bi in range(max(0, ci - 3), min(W - 1, ci + 3) + 1):
                c = bj * W + bi
                if built[c]:
                    d = dist_square(px, py, bi, bj)
                    if d < best: best = d
                for h in bucket[c]:
                    dx, dy = h[0] - px, h[1] - py
                    d = math.sqrt(dx * dx + dy * dy)
                    if d < best: best = d
        return best
    water_cells = [n for n in range(N) if L["water"][n] or L["built"][n]]
    sp, d_home, d_road, d_water = [0.0] * (N * SUB * SUB), [0.0] * N, [0.0] * N, [0.0] * N
    for j in range(H):
        for i in range(W):
            n = j * W + i
            for b in range(SUB):
                for a in range(SUB):
                    sp[(n * SUB + b) * SUB + a] = near_home(i, j, i + (a + 0.5) / SUB, j + (b + 0.5) / SUB)
            d_home[n] = near_home(i, j, i + 0.5, j + 0.5)
            d_road[n] = min(abs((j + 0.5 if r["axis"] == "h" else i + 0.5) - r["at"]) for r in D["roads"])
            bw = math.inf
            for cw in water_cells:
                wx, wy = cw % W + 0.5 - (i + 0.5), (cw - cw % W) // W + 0.5 - (j + 0.5)
                dw = math.sqrt(wx * wx + wy * wy)
                if dw < bw: bw = dw
            d_water[n] = bw
    L["sp"] = np.array(sp).reshape(N, SUB * SUB)
    L["dHome"], L["dRoad"], L["dWater"] = np.array(d_home), np.array(d_road), np.array(d_water)
    return L

def to_arrays(L):
    for k in ["water", "flood", "built", "ind", "prot", "district", "subc", "slope", "wind", "sun", "land", "q"]:
        L[k] = np.array(L[k])
    return L

R = to_arrays(distances(R))

def frac(L, setback_km):
    return (L["sp"] > setback_km).sum(axis=1) / (SUB * SUB)

def seq_sum(a):
    # a plain running sum, in the page's order (np.sum adds in a different order)
    t = 0.0
    for v in np.asarray(a, dtype=float).ravel(): t += float(v)
    return t

RL = REF["layers"]
for k in ["slope", "wind", "sun", "land", "dHome", "dRoad", "dWater"]:
    assert close(seq_sum(R[k]), RL["sums"][k]), (k, seq_sum(R[k]), RL["sums"][k])
for k, v in RL["counts"].items():
    assert int((R[k] > 0).sum()) == v, (k, int((R[k] > 0).sum()), v)
assert len(R["homes"]) == RL["homes"] and R["n_newer"] == RL["newer"]
for c in RL["cells"]:
    for k in ["slope", "wind", "sun", "land"]:
        assert close(R[k][c["n"]], c[k]), (c, k)
for key, s in [("m61", 0.061), ("m220", 0.22), ("m400", 0.4)]:
    assert close(seq_sum(frac(R, s)), RL["fracSum"][key])
print("layers, homes and setback shares: all equal to the page")
home_cells = {math.floor(h[1]) * W + math.floor(h[0]) for h in R["homes"]}
missed = sum(1 for n in home_cells if frac(R, 0.061)[n] == 1)
print(f"{missed} of the {len(home_cells)} cells with a home keep all 16 points at a 61 m setback: the sample misses their buffers")
assert (missed, len(home_cells)) == (127, 159)

In [ ]:
if HAVE_PLT:
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
    for a, (k, cmap, lab) in zip(ax, [("slope", "Oranges", "Slope, %"), ("wind", "Blues", "Wind at 100 m, m/s"), ("land", "Purples", "Land price, $ per acre")]):
        im = a.imshow(R[k].reshape(H, W), cmap=cmap); a.set_title(lab); a.set_xticks([]); a.set_yticks([])
        fig.colorbar(im, ax=a, shrink=0.8)
    plt.tight_layout(); plt.show()

## 2. Exclusions and the land left

An exclusion removes a cell outright. The order of the reasons matches the page: river, built-up land, industrial park (for solar and wind), substation, protected land, flood zone, slope above the rule. A cell that survives keeps its share outside the home setbacks, in km².

In [ ]:
T = D["types"]

def cfg(t, **over):
    c = {"exclusions": dict(T[t]["exclusions"]), "slopeMax": T[t]["slopeMax"], "setbackM": T[t]["setbackM"],
         "weights": dict(T[t]["weights"]), "anySub": False}
    for k, v in over.items():
        if k in ("exclusions", "weights"): c[k].update(v)
        else: c[k] = v
    return c

def exclude(L, t, c, area=1.0):
    E = c["exclusions"]
    reason = np.zeros(len(L["water"]), dtype=int)
    rules = [(1, L["water"] > 0), (2, L["built"] > 0), (3, (L["ind"] > 0) & (t != "dc")), (7, L["subc"] > 0),
             (4, (L["prot"] > 0) & E["protected"]), (5, (L["flood"] > 0) & E["flood"]), (6, (L["slope"] > c["slopeMax"]) & E["slope"])]
    for code, m in rules:
        reason = np.where((reason == 0) & m, code, reason)
    f = L["frac"](c["setbackM"] / 1000) if E["setback"] else np.ones(len(reason))
    buildable = np.where(reason == 0, area * f, 0.0)
    return {"reason": reason, "frac": f, "buildable": buildable, "area": seq_sum(buildable)}

R["frac"] = lambda s: frac(R, s)
R["cellKm"], R["W"], R["H"] = 1, W, H
c_solar = cfg("solar")
X = exclude(R, "solar", c_solar)
print("land left for a solar farm:", round(X["area"]), "km² of", N)
assert close(X["area"], REF["solar"]["eligibleKm2"])

## 3. Scores, the weighted overlay and the best blocks

Each criterion becomes a score from 0 to 1 (1 is best): resources rescaled from the region's lowest to its highest; distances and prices through a straight ramp between two stated ends. The suitability is

$$S = \frac{\sum_k w_k s_k}{\sum_k w_k}.$$

A site is a block of cells (2 km × 2 km for the solar farm).

- **Room.** A block has room when its buildable area $B = \sum_c a_c f_c$ reaches the land needed: $150 \times 1.29 \times 4.2 = 812.7$ acres $\approx 3.29$ km² (LBNL's 2019 median for tracking arrays, and 2019 tracking plants' 1.29 MW of panels per MW of inverters).
- **Score.** A block's score is the area-weighted mean of its cells' scores.
- **The short list.** The five best blocks that keep one cell apart; equal scores go to the more northern, then the more western block.
- **No weights.** With every weight at 0 nothing ranks the blocks, so `sites` stops with a message, as the page shows no short list.

In [ ]:
ACRES = 247.105

def xy(L, n):
    k = L["cellKm"]
    return ((n % L["W"]) + 0.5) * k, ((n - n % L["W"]) // L["W"] + 0.5) * k

def need_mw(t, c):
    return 0 if c["anySub"] else T[t]["mw"]

def sub_dist(x, y, mw):
    best = math.inf
    for s in D["substations"]:
        if s["headroomMW"] < mw: continue
        dx, dy = s["x"] - x, s["y"] - y
        best = min(best, math.sqrt(dx * dx + dy * dy))
    return best

def clamp01(v):
    return np.minimum(1.0, np.maximum(0.0, v))

def criteria(L, t, c):
    rp, n_all = T[t]["ramps"], len(L["water"])
    res = L["wind"] if t == "wind" else L["sun"]
    lo, hi = float(res.min()), float(res.max())
    dsub = np.array([sub_dist(*xy(L, n), need_mw(t, c)) for n in range(n_all)])
    out = {}
    for k in T[t]["criteria"]:
        if k == "res": out[k] = (res - lo) / (hi - lo)
        elif k == "sub": out[k] = 1 - clamp01((dsub - rp["sub"][0]) / (rp["sub"][1] - rp["sub"][0]))
        elif k == "road": out[k] = 1 - clamp01((L["dRoad"] - rp["road"][0]) / (rp["road"][1] - rp["road"][0]))
        elif k == "water": out[k] = 1 - clamp01((L["dWater"] - rp["water"][0]) / (rp["water"][1] - rp["water"][0]))
        elif k == "land": out[k] = 1 - clamp01((L["land"] - rp["land"][0]) / (rp["land"][1] - rp["land"][0]))
        elif k == "homes": out[k] = clamp01((L["dHome"] - rp["homes"][0]) / (rp["homes"][1] - rp["homes"][0]))
    return out

def need_km2(t):
    if t == "solar": return T[t]["mw"] * T[t]["dcac"] * T[t]["acresPerMWdc"] / ACRES
    if t == "wind": return T[t]["needShare"] * T[t]["blockKm"] ** 2
    return T[t]["needKm2"]

def blocks(L, t, c, X, CR):
    k = T[t]["blockKm"] // L["cellKm"]
    keys, req, out = T[t]["criteria"], need_km2(t), []
    Wd, Hd = L["W"], L["H"]
    for j in range(Hd - k + 1):
        for i in range(Wd - k + 1):
            B, m, lp, rs = 0.0, {kk: 0.0 for kk in keys}, 0.0, 0.0
            for b in range(k):
                for a in range(k):
                    n = (j + b) * Wd + i + a
                    bu = float(X["buildable"][n])
                    if not bu: continue
                    B += bu
                    for kk in keys: m[kk] += float(CR[kk][n]) * bu
                    lp += float(L["land"][n]) * bu
                    rs += float((L["wind"] if t == "wind" else L["sun"])[n]) * bu
            if B < req - 1e-9: continue
            out.append({"i": i, "j": j, "k": k, "B": B, "m": {kk: m[kk] / B for kk in keys}, "land": lp / B, "res": rs / B})
    return out

def block_score(bl, w, keys):
    t = tw = 0.0
    for kk in keys:
        t += w[kk] * bl["m"][kk]; tw += w[kk]
    return t / tw if tw > 0 else 0.0

def overlap(a, b, g=0):
    return a["i"] - g < b["i"] + b["k"] and b["i"] - g < a["i"] + a["k"] and a["j"] - g < b["j"] + b["k"] and b["j"] - g < a["j"] + a["k"]

def conn_cell(L, X, p):
    best, bd, c0 = -1, math.inf, (p["k"] - 1) / 2
    for b in range(p["k"]):
        for a in range(p["k"]):
            n, d = (p["j"] + b) * L["W"] + p["i"] + a, (a - c0) ** 2 + (b - c0) ** 2
            if X["buildable"][n] > 0 and d < bd: bd, best = d, n
    return best

def sites(L, t, c, X, CR, BL, n=5):
    keys, gap = T[t]["criteria"], T[t].get("gap", 0)
    # every weight 0: every score is 0 and only the tie rule would rank, so there is no short list (as on the page)
    assert sum(c["weights"][k] for k in keys) > 0, "every weight is 0: no criterion ranks the blocks, so there is no short list"

    ranked = sorted((dict(bl, score=block_score(bl, c["weights"], keys)) for bl in BL), key=lambda b: (-b["score"], b["j"], b["i"]))
    pick = []
    for b in ranked:
        if len(pick) == n: break
        if any(overlap(p, b, gap) for p in pick): continue
        pick.append(b)
    for r, p in enumerate(pick):
        p["id"], p["cell"] = chr(65 + r), conn_cell(L, X, p)
    return pick

CR = criteria(R, "solar", c_solar)
BL = blocks(R, "solar", c_solar, X, CR)
top = sites(R, "solar", c_solar, X, CR, BL)
assert len(BL) == REF["solar"]["blocks"]
for p, q in zip(top, REF["solar"]["top"]):
    assert (p["id"], p["i"], p["j"], p["cell"]) == (q["id"], q["i"], q["j"], q["cell"]) and close(p["score"], q["score"]), (p, q)
for p in top:
    print(p["id"], "block at", (p["i"], p["j"]), "score", round(100 * p["score"]), "buildable", round(p["B"], 2), "km²")

In [ ]:
if HAVE_PLT:
    keys, w = T["solar"]["criteria"], c_solar["weights"]
    S = sum(w[k] * CR[k] for k in keys) / sum(w.values())
    S = np.where(X["reason"] > 0, np.nan, S).reshape(H, W)
    fig, a = plt.subplots(figsize=(7, 4.8))
    a.imshow(S, cmap="Greens"); a.set_title("Solar suitability; excluded cells blank; the five best blocks")
    for p in top:
        a.add_patch(plt.Rectangle((p["i"] - 0.5, p["j"] - 0.5), p["k"], p["k"], fill=False, lw=2, ec="black"))
        a.text(p["i"] - 0.4, p["j"] - 0.8, p["id"], fontweight="bold")
    a.set_xticks([]); a.set_yticks([]); plt.show()

## 4. Weights are value choices

Draw 500 sets of five weights, find the best block for each, and count how often each default site is the winning block.

- **Every mix equally likely.** Sort four uniform numbers; the five gaps between 0, them and 1 are the weights.
- **A win.** It counts for a site only when the winner is that very block. A winner that merely overlaps a site, a cell or two away, is another block (the cell also counts those).
- **Fast.** The block scores are linear in the weights, so one matrix product scores every block at once.

In [ ]:
def shares(L, t, c, top, BL, draws, seed):
    keys = T[t]["criteria"]
    Mx = np.array([[bl["m"][k] for k in keys] for bl in BL])     # blocks × criteria
    rng, wins, other, near = mulberry32(seed), {p["id"]: 0 for p in top}, 0, 0
    for _ in range(draws):
        u = sorted(rng() for _ in range(len(keys) - 1))
        w = np.diff([0.0] + u + [1.0])
        sc = Mx @ w                                              # the weights sum to 1
        best = max(range(len(BL)), key=lambda q: (sc[q], -BL[q]["j"], -BL[q]["i"]))
        hit = next((p for p in top if (p["i"], p["j"]) == (BL[best]["i"], BL[best]["j"])), None)
        if hit: wins[hit["id"]] += 1
        else:
            other += 1
            if any(overlap(p, BL[best]) for p in top): near += 1
    return wins, other, near

S_ = D["sensitivity"]
wins, other, near = shares(R, "solar", c_solar, top, BL, S_["draws"], S_["seed"])
print({k: f"{v / S_['draws']:.0%}" for k, v in wins.items()}, "another block:", f"{other / S_['draws']:.0%}", f"({near} of those {other} overlap a default site)")
assert wins == REF["shares"]["wins"] and other == REF["shares"]["other"] and near == REF["shares"]["near"]

## 5. A cost surface and the least-cost path

Each cell gets a cost per km of new line, in four parts:

- **The baseline.** NREL's 138 kV line with MISO's costs (also used for SPP): \$1,615,100 a mile in 2022 dollars, brought to 2025 dollars with BEA's GDP deflator, a price index ($\times\,128.893 / 118.026$).
- **Land cover.** Times a multiplier: farmland 1.0, rural fringe 1.1, built-up 1.2, hilly 1.1, flood zone or river 1.8.
- **Protected land.** Either impassable, or allowed at NREL's highest mitigation cost, \$38,000 a mile in 2022 dollars (about \$41,500 in 2025 dollars).
- **A step.** Between neighbouring cells, the mean of their two costs times the step's length: 1 km, or $\sqrt{2}$ km on a diagonal.

**Dijkstra's algorithm**, started from every substation with room for 150 MW, finds the cheapest route from any cell.

In [ ]:
K = D["costs"]
KM_PER_MILE = 1.609344
IX = K["deflator"]["index"]

def usd(amount, year):
    # an amount of a given year in 2025 dollars (BEA's GDP deflator), in the page's order of operations
    return amount * (IX[str(K["year"])] / IX[str(year)])

print("line on farmland:", f"${usd(K['lineUSDperMile'], K['lineYear']) / KM_PER_MILE:,.0f} per km in {K['year']} dollars")
NB = [(0, -1), (0, 1), (-1, 0), (1, 0), (-1, -1), (1, -1), (-1, 1), (1, 1)]

def surface(L, protected):
    base, mit, Mu = usd(K["lineUSDperMile"], K["lineYear"]) / KM_PER_MILE, usd(K["mitigationUSDperMile"], K["mitigationYear"]) / KM_PER_MILE, K["mult"]
    c, cls = np.zeros(N), []
    for n in range(N):
        mult, cl = Mu["farmland"], "farmland"
        if L["district"][n] == 1 and Mu["fringe"] > mult: mult, cl = Mu["fringe"], "fringe"
        if (L["built"][n] or L["ind"][n]) and Mu["built"] > mult: mult, cl = Mu["built"], "built"
        if L["slope"][n] > K["hillyFrom"] and Mu["hilly"] > mult: mult, cl = Mu["hilly"], "hilly"
        if (L["flood"][n] or L["water"][n]) and Mu["wet"] > mult: mult, cl = Mu["wet"], "wet"
        cls.append("protected" if L["prot"][n] else cl)
        c[n] = math.inf if (L["prot"][n] and protected == "forbid") else base * mult + (mit if L["prot"][n] else 0)
    return c, cls

def dijkstra(c, need):
    dist, pred, src, done, h = [math.inf] * N, [-1] * N, [-1] * N, [False] * N, []
    for k, s in enumerate(D["substations"]):
        n = math.floor(s["y"]) * W + math.floor(s["x"])
        if s["headroomMW"] >= need and dist[n] > 0 and c[n] < math.inf:
            dist[n], src[n] = 0.0, k; heapq.heappush(h, (0.0, n))
    while h:
        _, u = heapq.heappop(h)
        if done[u]: continue
        done[u] = True
        ui, uj = u % W, u // W
        for z, (di, dj) in enumerate(NB):
            vi, vj = ui + di, uj + dj
            if not (0 <= vi < W and 0 <= vj < H): continue
            v = vj * W + vi
            if done[v] or c[v] == math.inf: continue
            nd = dist[u] + (c[u] + c[v]) / 2 * (1 if z < 4 else math.sqrt(2))
            if nd < dist[v]:
                dist[v], pred[v], src[v] = nd, u, src[u]; heapq.heappush(h, (nd, v))
    return dist, pred, src

def route(dist, pred, src, cls, cell):
    cells, km, parts, n = [cell], 0.0, {}, cell
    while pred[n] >= 0:
        m = pred[n]
        ln = math.sqrt(2) if (n % W != m % W and n // W != m // W) else 1
        km += ln
        for x in (n, m): parts[cls[x]] = parts.get(cls[x], 0) + ln / 2
        cells.append(m); n = m
    return {"cells": cells, "km": km, "cost": dist[cell], "sub": D["substations"][src[cell]]["id"], "parts": parts}

c_wind = cfg("wind")
Xw = exclude(R, "wind", c_wind); CRw = criteria(R, "wind", c_wind); BLw = blocks(R, "wind", c_wind, Xw, CRw)
top_w = sites(R, "wind", c_wind, Xw, CRw, BLw)
paths = {}
for prot in ["forbid", "allow"]:
    cs, cls = surface(R, prot)
    dist, pred, src = dijkstra(cs, T["wind"]["mw"])
    rt = route(dist, pred, src, cls, top_w[0]["cell"])
    paths[prot] = (dist, pred, src, cls, rt)
    ref = REF["corridor"][prot]
    assert rt["sub"] == ref["sub"] and close(rt["km"], ref["km"]) and abs(rt["cost"] - ref["cost"]) < 1 and len(rt["cells"]) == ref["cells"], (prot, rt["km"], ref)
    print(prot, "protected land:", f"{rt['km']:.1f} km to {rt['sub']}, ${rt['cost'] / 1e6:.1f} million;",
          ", ".join(f"{v:.1f} km {k}" for k, v in rt["parts"].items()))

Forbidding protected land is a **rule**; pricing it at about \$41,500 a mile (NREL's \$38,000 in 2022 dollars) is a **weight** so small that the line cuts straight through. That is the difference between an exclusion and a criterion.

## 6. The sites passed to a cost model

Each short-listed site gets two numbers, kept in separate columns.

| Column | What it counts |
|---|---|
| Up front, 2025 dollars | the line along its least-cost path; an access road to the nearest paved road (\$600,000 a km, set for teaching); land at the block's mean price (the solar arrays' 812.7 acres; for wind, the permanent footprint, 0.3 ha per MW) |
| Energy, MWh a year | $\text{MW} \times \text{capacity factor} \times 8{,}760$ h, with EIA's 2025 fleet capacity factors at the region's average sunshine and wind (solar 24.0%, wind 33.8%; the page reads `cf_targets.json`) |

One is dollars paid once, the other MWh a year: no rate makes them addable. To compare sites on both, value the energy at an electricity price and discount the cash flows (lesson A1). Or spread the up-front cost over the years as an annualized cost and divide it by the annual energy: a cost per MWh.

In [ ]:
CF_SOLAR, CF_WIND = 0.240, 0.338     # EIA, Electric Power Monthly, Table 6.07.B, U.S. utility-scale fleets, 2025

def site_cost(L, t, p, dist, pred, src, cls):
    Tt, out = T[t], {}
    rt = route(dist, pred, src, cls, p["cell"]) if dist[p["cell"]] < math.inf else None
    out["connect"], out["sub"] = (rt["cost"], rt["sub"]) if rt else (math.inf, None)
    out["road"] = float(L["dRoad"][p["cell"]]) * usd(K["roadUSDperKm"], K["roadYear"])
    out["water"] = float(L["dWater"][p["cell"]]) * usd(K["waterUSDperKm"], K["waterYear"]) if t == "dc" else 0.0
    acres = Tt["mw"] * Tt["dcac"] * Tt["acresPerMWdc"] if t == "solar" else Tt["mw"] * Tt["directHaPerMW"] * ACRES / 100 if t == "wind" else Tt["needKm2"] * ACRES
    out["land"] = acres * p["land"]
    out["upfront"] = out["connect"] + out["road"] + out["water"] + out["land"]
    if t != "dc":
        cf = CF_SOLAR * p["res"] / float(np.mean(L["sun"])) if t == "solar" else CF_WIND + Tt["cfPerMs"] * (p["res"] - float(np.mean(L["wind"])))
        out["mwh"] = Tt["mw"] * cf * 8760
    return out

def check_costs(t, L, top, P, ref):
    for p, q in zip(top, ref["top"]):
        k = site_cost(L, t, p, *P)
        for f in ["connect", "land", "road", "water", "upfront"]:
            assert abs(k[f] - q[f]) <= 1, (t, p["id"], f, k[f], q[f])
        assert k["sub"] == q["sub"] and ("mwh" not in q or abs(k["mwh"] - q["mwh"]) <= 1)
        up = next(s["headroomMW"] for s in D["substations"] if s["id"] == k["sub"]) < T[t]["mw"]   # a substation without that much spare capacity
        print(f"{t} {p['id']}: score {round(100 * p['score'])}, up front ${k['upfront'] / 1e6:.1f} million ({k['sub']})" + (" + an upgrade and a wait, not priced: not comparable" if up else "") + (f", {k['mwh'] / 1000:,.0f} GWh a year" if "mwh" in k else ""))

cs, cls = surface(R, "forbid")
P150 = dijkstra(cs, 150) + (cls,)
check_costs("solar", R, top, P150, REF["solar"])
check_costs("wind", R, top_w, paths["forbid"][:4], REF["wind"])

### The data-center campus

A data center here is a load and a building: Prairie City's campus, whose contract MW (the most it may draw) comes from the shared city data.

- **Its needs** (set for teaching): 0.5 km² of land, water and a road, and a 300 m noise buffer from homes.
- **Its connection.** It builds its own substation and a line to one of the map's transmission substations (S1 to S4, not the districts' own) with that much spare capacity.
- **Any substation instead.** A site can then land next to one that would need an upgrade first, and a wait.

In [ ]:
print("campus contract:", T["dc"]["mw"], "MW (prairie-city.json)")
for any_sub, key in [(False, "dc"), (True, "dcAny")]:
    c_dc = cfg("dc", anySub=any_sub)
    Xd = exclude(R, "dc", c_dc); CRd = criteria(R, "dc", c_dc); BLd = blocks(R, "dc", c_dc, Xd, CRd)
    top_d = sites(R, "dc", c_dc, Xd, CRd, BLd)
    assert [(p["i"], p["j"]) for p in top_d] == [(q["i"], q["j"]) for q in REF[key]["top"]]
    Pd = dijkstra(cs, need_mw("dc", c_dc)) + (cls,)
    check_costs("dc", R, top_d, Pd, REF[key])

## 7. Two limits: cell size and the data's date

**2 km cells.** Average each 2 × 2 group of cells (numbers by their mean, classes by a majority of at least half), then screen again with one 2 km cell per solar block. Thin features such as the river vanish; averages hide steep bluffs.

**A dated homes layer.** The homes layer is dated 2019; 36 homes were built since. Add them and screen again.

In [ ]:
def coarsen(L, f=2):
    Wc, Hc = W // f, H // f
    C2 = {"W": Wc, "H": Hc, "cellKm": f, "homes": L["homes"]}
    for k in ["water", "flood", "built", "ind", "prot", "subc", "district"]: C2[k] = np.zeros(Wc * Hc, dtype=int)
    for k in ["slope", "wind", "sun", "land", "dHome", "dRoad", "dWater"]: C2[k] = np.zeros(Wc * Hc)
    half = f * f / 2
    for J in range(Hc):
        for I in range(Wc):
            n = J * Wc + I
            ms = [(J * f + b) * W + I * f + a for b in range(f) for a in range(f)]
            cnt = lambda k: sum(1 for m in ms if L[k][m])
            C2["water"][n] = cnt("water") >= half
            C2["flood"][n] = (not C2["water"][n]) and sum(1 for m in ms if L["flood"][m] or L["water"][m]) >= half
            C2["built"][n] = (not C2["water"][n]) and cnt("built") >= half
            C2["ind"][n] = (not C2["water"][n]) and cnt("ind") >= half
            pk = [L["prot"][m] for m in ms if L["prot"][m]]
            C2["prot"][n] = pk[-1] if len(pk) >= half else 0
            C2["subc"][n] = any(L["subc"][m] for m in ms)
            C2["district"][n] = max(L["district"][m] for m in ms)
            for k in ["slope", "wind", "sun", "land"]:
                t = 0.0
                for m in ms: t += float(L[k][m])
                C2[k][n] = t / (f * f)
            x, y = (I + 0.5) * f, (J + 0.5) * f
            C2["dRoad"][n] = min(abs((y if r["axis"] == "h" else x) - r["at"]) for r in D["roads"])
            bh = CAP
            for h in L["homes"]: bh = min(bh, math.sqrt((h[0] - x) ** 2 + (h[1] - y) ** 2))
            for m in range(N):
                ii, jj = m % W, m // W
                if L["built"][m]: bh = min(bh, dist_square(x, y, ii, jj))
            C2["dHome"][n] = bh
            bw = math.inf
            for m in range(N):
                if L["water"][m] or L["built"][m]:
                    ii, jj = m % W, m // W
                    bw = min(bw, math.sqrt((ii + 0.5 - x) ** 2 + (jj + 0.5 - y) ** 2))
            C2["dWater"][n] = bw
    def cfrac(s):
        ff = frac(L, s)
        return np.array([sum(ff[(J * f + b) * W + I * f + a] for b in range(f) for a in range(f)) / (f * f) for J in range(Hc) for I in range(Wc)])
    C2["frac"] = cfrac
    return C2

C2 = coarsen(R)
X2 = exclude(C2, "solar", c_solar, area=4.0)
CR2 = criteria(C2, "solar", c_solar); BL2 = blocks(C2, "solar", c_solar, X2, CR2); top2 = sites(C2, "solar", c_solar, X2, CR2, BL2)
print("2 km cells:", round(X2["area"]), "km² left, against", round(X["area"]), "at 1 km")
assert close(X2["area"], REF["coarse"]["eligibleKm2"]) and [(p["i"], p["j"]) for p in top2] == [(q["i"], q["j"]) for q in REF["coarse"]["top"]]

Rn = to_arrays(distances(build(newer=True)))
Rn["frac"] = lambda s: frac(Rn, s); Rn["cellKm"], Rn["W"], Rn["H"] = 1, W, H
Xn = exclude(Rn, "solar", c_solar); CRn = criteria(Rn, "solar", c_solar); BLn = blocks(Rn, "solar", c_solar, Xn, CRn); topn = sites(Rn, "solar", c_solar, Xn, CRn, BLn)
print("with the newer homes:", round(Xn["area"]), "km² left; top five", [(p["i"], p["j"]) for p in topn])
assert close(Xn["area"], REF["newer"]["eligibleKm2"]) and [(p["i"], p["j"]) for p in topn] == [(q["i"], q["j"]) for q in REF["newer"]["top"]]
print("site A at", (top[0]["i"], top[0]["j"]), "in the new top five:", any((p["i"], p["j"]) == (top[0]["i"], top[0]["j"]) for p in topn))

## 8. Cleaning raw layers before a screen (simulated raw layers)

Public layers rarely arrive ready to stack. This section makes two **simulated raw layers** from the page's own layers, with the faults real downloads often have, then cleans them and checks the result against the page. Nothing is downloaded: the raw layers are generated here from the seed and are labelled simulated.

1. **A homes table in degrees.** Points as longitude and latitude, with duplicate rows, a few rows whose latitude and longitude are swapped, and records with no coordinates. Clean it, project it onto the km grid, and check the distance error of the projection.
2. **A wind raster on another grid.** 2 km cells whose left edge lies 2 km west of the region, with a no-data value of $-9{,}999$ in some cells. Mask the no-data cells, align the raster to the 1 km grid, and flag the cells that had to be filled.

The region is pinned at a made-up place, its north-west corner at 39.0° N, 98.0° W, only so that degrees exist to convert; Prairie City stays fictional.

In [ ]:
# --- the simulated raw homes table (generated from the page's homes; seeded faults)
LAT0, LON0, R_EARTH = 39.0, -98.0, 6371.0088          # made-up placement of the north-west corner; mean Earth radius, km
LAT_MID = LAT0 - math.degrees((H / 2) / R_EARTH)          # the region's middle latitude
KX = R_EARTH * math.cos(math.radians(LAT_MID)) * math.pi / 180   # km per degree of longitude at the middle latitude
KY = R_EARTH * math.pi / 180                                       # km per degree of latitude

def to_deg(x, y):                    # km on the grid (x east, y south of the north-west corner) -> (lon, lat)
    return LON0 + x / KX, LAT0 - y / KY

def to_km(lon, lat):                 # the local projection: (lon, lat) -> km on the grid
    return (lon - LON0) * KX, (LAT0 - lat) * KY

rng = mulberry32(SEED + 900)
raw = [{"id": f"H{k:03d}", "lon": round(to_deg(h[0], h[1])[0], 6), "lat": round(to_deg(h[0], h[1])[1], 6)} for k, h in enumerate(R["homes"])]
for k in sorted({math.floor(rng() * len(raw)) for _ in range(6)})[:5]:
    raw.append(dict(raw[k]))                                       # duplicate rows
swapped = sorted({math.floor(rng() * len(R["homes"])) for _ in range(4)})[:3]
for k in swapped:
    raw[k]["lon"], raw[k]["lat"] = raw[k]["lat"], raw[k]["lon"]    # latitude and longitude swapped
for k in range(4):
    raw.append({"id": f"P{k:03d}", "lon": None, "lat": None})      # records with no coordinates (not yet located)
print(len(raw), "raw rows (simulated); first three:", raw[:3])

In [ ]:
# --- clean it, in this order: missing values, coordinate ranges (fix plausible swaps), then duplicates (after the
# coordinates are fixed, so a swapped row and its copy are recognised as one); then project onto the grid
lon_w, lat_n = to_deg(0, 0)
lon_e, lat_s = to_deg(W, H)
inside = lambda lon, lat: lon_w <= lon <= lon_e and lat_s <= lat <= lat_n
seen, clean, report = set(), [], {"duplicates": 0, "swapped": 0, "missing": 0, "outside": 0}
for r in raw:
    if r["lon"] is None or r["lat"] is None:
        report["missing"] += 1; continue                           # cannot be placed: drop, and say so
    lon, lat = r["lon"], r["lat"]
    if not inside(lon, lat) and inside(lat, lon):
        lon, lat = lat, lon; report["swapped"] += 1                 # a plausible swap: fix it, and say so
    if not inside(lon, lat):
        report["outside"] += 1; continue
    if (r["id"], lon, lat) in seen:
        report["duplicates"] += 1; continue
    seen.add((r["id"], lon, lat))
    clean.append(to_km(lon, lat))
print("cleaning report:", report, "->", len(clean), "homes")
assert report == {"duplicates": 5, "swapped": 3, "missing": 4, "outside": 0} and len(clean) == len(R["homes"])
err = max(math.hypot(a[0] - h[0], a[1] - h[1]) for a, h in zip(clean, R["homes"]))
print(f"largest position error after the round trip: {1000 * err:.2f} m (6-decimal degrees are about 0.1 m)")
assert err < 0.001
counts = np.zeros(N, dtype=int); page = np.zeros(N, dtype=int)
for x, y in clean: counts[math.floor(y) * W + math.floor(x)] += 1
for h in R["homes"]: page[math.floor(h[1]) * W + math.floor(h[0])] += 1
assert (counts == page).all()
print("homes per cell after cleaning = the page's homes layer, cell by cell")

In [ ]:
# --- how far off are distances on this projection? Compare with the great-circle (haversine) distance on a sphere
def haversine(lon1, lat1, lon2, lat2):
    p1, p2, dl, dp = math.radians(lat1), math.radians(lat2), math.radians(lon2 - lon1), math.radians(lat2 - lat1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R_EARTH * math.asin(math.sqrt(a))

worst = 0.0
for (x1, y1, x2, y2) in [(0, 0, W, 0), (0, H, W, H), (0, 0, W, H), (W / 2, 0, W / 2, H), (1, 1, 1.22, 1), (40, 28, 40.22, 28)]:
    plane = math.hypot(x2 - x1, y2 - y1)
    sphere = haversine(*to_deg(x1, y1), *to_deg(x2, y2))
    worst = max(worst, abs(plane - sphere) / sphere)
    print(f"{plane:8.3f} km on the grid, {sphere:8.3f} km on the sphere ({100 * (plane - sphere) / sphere:+.3f}%)")
print(f"largest relative error: {100 * worst:.2f}%: {220 * worst:.2f} m on a 220 m setback")
assert worst < 0.005

On a region 44 km across, this simple projection is within a fraction of a percent of the great-circle distance: fine for setbacks and line lengths here. Over a state or a country it is not; an equal-area projection keeps **areas** true but stretches **distances** away from its standard lines, so check the distance error where you measure, use a local projection that keeps distances close (such as the area's UTM zone), or compute geodesic distances.

In [ ]:
# --- the simulated raw wind raster: 2 km cells, left edge 2 km west of the region, no-data cells
NODATA, GT = -9999.0, {"x0": -2.0, "y0": 0.0, "cell": 2.0}       # the raster's geotransform: left edge, top edge, cell size (km)
rows_, cols_ = H // 2, W // 2 + 1
wind_raw = np.full((rows_, cols_), NODATA)
for J in range(rows_):
    for I in range(1, cols_):                                      # column 0 lies west of the region: no data there
        ms = [(2 * J + b) * W + 2 * (I - 1) + a for b in range(2) for a in range(2)]
        t = 0.0
        for m in ms: t += float(R["wind"][m])
        wind_raw[J, I] = t / 4
holes = [(3, 5), (7, 12), (10, 20)]                                # cells lost upstream (fixed for the exercise)
for J, I in holes: wind_raw[J, I] = NODATA
print("naive mean of the raw raster:", round(float(wind_raw.mean()), 1), "m/s: the no-data value leaked in")
print("mean of the valid cells:     ", round(float(wind_raw[wind_raw != NODATA].mean()), 2), "m/s")
assert wind_raw.mean() < 0 < wind_raw[wind_raw != NODATA].mean()

In [ ]:
# --- align it to the 1 km grid: a cell's centre falls in raster column floor((x - x0) / cell), row floor((y - y0) / cell)
w = np.where(wind_raw == NODATA, np.nan, wind_raw)
aligned, naive = np.zeros(N), np.zeros(N)
for n in range(N):
    x, y = n % W + 0.5, n // W + 0.5
    aligned[n] = w[math.floor((y - GT["y0"]) / GT["cell"]), math.floor((x - GT["x0"]) / GT["cell"])]
    naive[n] = w[math.floor(y / GT["cell"]), math.floor(x / GT["cell"])]       # forgets the 2 km offset
flag = np.isnan(aligned)
truth = np.array([C2["wind"][(n // W // 2) * (W // 2) + (n % W) // 2] for n in range(N)])  # section 7's 2 km means
ok_cells = ~flag
assert np.allclose(aligned[ok_cells], truth[ok_cells], atol=1e-12) and int(flag.sum()) == 4 * len(holes)
off = np.nanmax(np.abs(naive - truth))
print("aligned with the offset: equal to the 2 km means of the page's wind layer;", int(flag.sum()), "cells flagged as no data")
print(f"ignoring the offset shifts every value by one 2 km column: up to {off:.2f} m/s wrong")
assert off > 0.1
# fill the flagged cells from the valid raster cells beside them (left, right, above, below), and keep the flag
filled = aligned.copy()
for J, I in holes:
    nb = [w[J + dj, I + di] for dj, di in [(0, -1), (0, 1), (-1, 0), (1, 0)] if 0 <= J + dj < rows_ and 0 <= I + di < cols_ and not np.isnan(w[J + dj, I + di])]
    for b in range(2):
        for a in range(2):
            filled[(2 * J + b) * W + 2 * (I - 1) + a] = sum(nb) / len(nb)
print(f"filled {int(flag.sum())} cells; largest fill error against the true layer: {np.max(np.abs(filled - R['wind'])[flag]):.2f} m/s")
print("Record both in the data note: which cells were filled, and how.")

## 9. With real data

A real screen replaces each made-up layer with a public one. This notebook downloads nothing; these are the places to look, as of November 2026:

| Layer here | Public source | What to watch |
|---|---|---|
| Slope | USGS [3D Elevation Program](https://www.usgs.gov/3d-elevation-program) (elevation; derive slope) | compute slope on the fine grid, then aggregate |
| Flood zones | FEMA [National Flood Hazard Layer](https://www.fema.gov/flood-maps/national-flood-hazard-layer) | coverage and map dates differ by county |
| Protected land | USGS [PAD-US](https://www.usgs.gov/programs/gap-analysis-project/science/pad-us-data-overview), the national inventory of protected areas | which protection classes your rules exclude |
| Roads | U.S. Census Bureau [TIGER/Line shapefiles](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html) | the vintage year of each release |
| Power plants and lines | EIA [U.S. Energy Atlas](https://atlas.eia.gov/); EIA's [answer on plant and line locations](https://www.eia.gov/tools/faqs/faq.php?id=567&t=3) (plants from Form EIA-860; lines from the federal HIFLD platform) | substation locations and spare capacity are not published there: ask the utility, or read its hosting-capacity maps |
| Wind and sunshine | NLR (formerly NREL) [WIND Toolkit](https://www.nlr.gov/grid/wind-toolkit) and [solar resource data](https://www.nlr.gov/grid/solar-resource/renewable-resource-data), including the NSRDB | hub height and the years averaged |
| Land price | USDA's annual *Land Values* summary (named here, not linked) | state and regional averages, not parcel prices |

**Getting the layers onto one grid.** Real layers come as rasters (grids) and vectors (points, lines, polygons) in different coordinate systems, with the faults section 8 cleaned.

- **Areas.** Measure them on an equal-area projection (for the contiguous U.S., an Albers equal-area projection such as EPSG:5070).
- **Distances.** An equal-area projection does not keep them true. For setbacks and line lengths, check the distance error where you measure, use a projection that keeps local distances close (such as the area's UTM zone), or compute geodesic distances.
- **One grid.** Rasterize the vectors onto your grid; check every layer's date and resolution, and record them.
- **Resolution.** A screen at 1 km cells misses setbacks of a few hundred metres; NREL's supply curves check setbacks on 90 m cells ([NREL, 2025](https://docs.nlr.gov/docs/fy25osti/91900.pdf), p. 19).

## 10. Practice

1. **Stricter rules.** Screen the solar farm with NREL's limited case (slopes over 5% and 152 m from buildings). How much land is left, and does site A survive? *(Check: the page's step 2 with "Stricter rules" on.)*
2. **Your own weights.** Find weights that put site A in the far west, or say why none can. *(Hint: which criteria are best there?)*
3. **Another rule for the line.** Give built-up land a multiplier of 3.0 (SCE's urban value in NREL's Table 12) and reroute the wind farm's line. Does it change? Then redo the cost in 2022 dollars: by how much does the deflator move it?
4. **A gross supply curve.** Take every solar block with room, keep blocks that do not overlap (best score first), price each with `site_cost`, and sort them by up-front cost per MW. Plot cumulative MW against cost per MW: a *gross* candidate supply curve.
   - Each site was screened alone, so many share one substation's spare capacity: the curve's cumulative MW is not capacity the grid could connect at once.
   - Before a planning model such as lesson M3's uses it, assign each site its substation and cap each substation's total at its spare capacity.
5. **Write-up.** In five sentences: which of the four limits (weights, exclusion against weighting, resolution, data vintage) changed your short list most, and what would you check on the ground?

In [ ]:
# Practice 4, a start: the non-overlapping solar blocks with room, priced, cheapest per MW first
ranked = sorted((dict(bl, score=block_score(bl, c_solar["weights"], T["solar"]["criteria"])) for bl in BL), key=lambda b: (-b["score"], b["j"], b["i"]))
chosen = []
for b in ranked:
    if any(overlap(p, b, 1) for p in chosen): continue
    b["cell"] = conn_cell(R, X, b); chosen.append(b)
priced = sorted(((site_cost(R, "solar", b, *P150)["upfront"] / T["solar"]["mw"], b) for b in chosen), key=lambda z: z[0])
print(len(priced), "sites; cheapest five, $ per MW up front:", [round(z[0]) for z in priced[:5]])
# the grid's side: every site connects to a substation with room for one farm, but the sites share them
by_sub = {}
for z in priced:
    sub = site_cost(R, "solar", z[1], *P150)["sub"]
    by_sub[sub] = by_sub.get(sub, 0) + T["solar"]["mw"]
room = {s["id"]: s["headroomMW"] for s in D["substations"]}
print(f"gross: {len(priced) * T['solar']['mw']:,} MW of sites; their substations' spare capacity: {sum(room[k] for k in by_sub):,} MW")
for k, mw in sorted(by_sub.items()):
    print(f"  {k}: {mw:,} MW of sites against {room[k]} MW spare")
if HAVE_PLT:
    mw = np.arange(1, len(priced) + 1) * T["solar"]["mw"]
    plt.step(mw, [z[0] / 1000 for z in priced], where="post"); plt.xlabel("Cumulative MW"); plt.ylabel("Site cost, $ thousand per MW (2025)")
    plt.title("A gross site-cost supply curve for solar farms (fictional region; shared substations not capped)"); plt.show()